# 01. Raw data audit

Check the bank, failure and economic data files. Record the input files and prepare the shared configuration. Raw data remain unchanged.


## 1. Mount Google Drive and import packages


In [ ]:
# Mount Google Drive and import packages

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from pprint import pformat
import calendar
import csv
import hashlib
import importlib.util
import json
import re
import sys
import zipfile

import numpy as np
import pandas as pd
from openpyxl import load_workbook
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("Google Drive mounted successfully.")


## 2. Define read-only raw paths and isolated analysis output paths


In [ ]:
# Define read-only raw paths and isolated main analysis output paths

PROJECT_ROOT = Path("/content/drive/MyDrive/Data Final")

# Existing raw inputs (read only)
RAW_ROOT = PROJECT_ROOT / "Raw Data"
FFIEC_RAW_DIR = RAW_ROOT / "FFIEC"
FDIC_RAW_DIR = RAW_ROOT / "FDIC"
MACRO_RAW_DIR = RAW_ROOT / "Macro"
FDIC_FILE = FDIC_RAW_DIR / "FDIC Data.xlsx"

EXPECTED_MACRO_FILES = (
    "real_gdp_raw.csv",
    "unemployment_rate_raw.csv",
    "effective_federal_funds_rate_raw.csv",
    "house_price_index_raw.csv",
)

# All new results are isolated from earlier analysis.
V2_ROOT = PROJECT_ROOT / "Version 2 Main Analysis"
V2_OUTPUT_ROOT = V2_ROOT / "Notebook Outputs"
V2_PROCESSED_DATA_DIR = V2_ROOT / "Processed Data"
NB01_OUTPUT_DIR = V2_OUTPUT_ROOT / "01_Raw_Data_Audit"
CONFIG_PATH = V2_ROOT / "config.py"

# Existing earlier analysis locations are declared for comparison only and must remain read only.
V1_OUTPUT_ROOT = PROJECT_ROOT / "Notebook Outputs"
V1_PROCESSED_DATA_DIR = PROJECT_ROOT / "Processed Data"

def is_relative_to(child, parent):
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

assert is_relative_to(V2_ROOT, PROJECT_ROOT)
assert not is_relative_to(V2_ROOT, RAW_ROOT)
assert V2_OUTPUT_ROOT != V1_OUTPUT_ROOT
assert V2_PROCESSED_DATA_DIR != V1_PROCESSED_DATA_DIR
assert is_relative_to(NB01_OUTPUT_DIR, V2_ROOT)
assert is_relative_to(CONFIG_PATH, V2_ROOT)

NB01_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
V2_PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw inputs (read only): {RAW_ROOT}")
print(f"V2 root:               {V2_ROOT}")
print(f"Notebook 01 outputs:   {NB01_OUTPUT_DIR}")
print("V1 output paths will not be written by this notebook.")


## 3. Lock the analysis timing policy

The key set and values below reproduce the authoritative definition in the analysis notebook sequence, Phase 6 cells A56–A79. The ordered-key assertion prevents any key from being added, renamed or omitted.


In [ ]:
# Lock the authoritative main analysis timing policy

EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS = (
    "analysis_role",
    "analysis_version",
    "bank_predictor_source",
    "macro_predictor_source",
    "target_window",
    "drop_missing_exact_predecessor",
    "repeat_development_eda",
    "repeat_all_validation_folds",
    "retune_hyperparameters",
    "reselect_validation_threshold",
    "archive_version_1_as_sensitivity",
    "reference_calendar_start",
    "reuse_version_1_predictor_manifest_by_default",
    "reuse_version_1_candidate_grids",
    "main_analysis_choice_is_unconditional_on_performance",
    "decision_basis",
    "decision_recorded_date",
    "require_common_sample_v1_comparison",
    "bank_predictor_lag_quarters",
    "archived_v1_read_only",
    "bank_lag_match_policy",
    "effective_horizon_from_predictor_date",
)

VERSION_2_MAIN_TIMING_POLICY = {
    "analysis_role": "main",
    "analysis_version": "v2_bank_lag1_main",
    "bank_predictor_source": "exact_t_minus_1",
    "macro_predictor_source": "t_minus_1",
    "target_window": "(t, t_plus_365_days]",
    "drop_missing_exact_predecessor": True,
    "repeat_development_eda": True,
    "repeat_all_validation_folds": True,
    "retune_hyperparameters": True,
    "reselect_validation_threshold": True,
    "archive_version_1_as_sensitivity": True,
    "reference_calendar_start": "2001-06-30",
    "reuse_version_1_predictor_manifest_by_default": True,
    "reuse_version_1_candidate_grids": True,
    "main_analysis_choice_is_unconditional_on_performance": True,
    "decision_basis": "reporting_availability_and_supervisor_feedback",
    "decision_recorded_date": "2026-08-13",
    "require_common_sample_v1_comparison": True,
    "bank_predictor_lag_quarters": 1,
    "archived_v1_read_only": True,
    "bank_lag_match_policy": "exact_calendar_quarter",
    "effective_horizon_from_predictor_date": (
        "realised_from_source_report_date_to_failure_date"
    ),
}

assert tuple(VERSION_2_MAIN_TIMING_POLICY) == (
    EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS
), "Authoritative V2 policy keys differ in name, number or order."
assert set(VERSION_2_MAIN_TIMING_POLICY) == set(
    EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS
), "Authoritative V2 policy key set is not exact."

# Convenience aliases are derived from the dictionary, never independent literals.
ANALYSIS_ROLE = VERSION_2_MAIN_TIMING_POLICY["analysis_role"]
ANALYSIS_VERSION = VERSION_2_MAIN_TIMING_POLICY["analysis_version"]
BANK_PREDICTOR_SOURCE = VERSION_2_MAIN_TIMING_POLICY["bank_predictor_source"]
MACRO_PREDICTOR_SOURCE = VERSION_2_MAIN_TIMING_POLICY["macro_predictor_source"]
TARGET_WINDOW = VERSION_2_MAIN_TIMING_POLICY["target_window"]
REFERENCE_CALENDAR_START = VERSION_2_MAIN_TIMING_POLICY["reference_calendar_start"]
BANK_PREDICTOR_LAG_QUARTERS = VERSION_2_MAIN_TIMING_POLICY["bank_predictor_lag_quarters"]
ARCHIVED_V1_READ_ONLY = VERSION_2_MAIN_TIMING_POLICY["archived_v1_read_only"]

assert ANALYSIS_ROLE == VERSION_2_MAIN_TIMING_POLICY["analysis_role"]
assert ANALYSIS_VERSION == VERSION_2_MAIN_TIMING_POLICY["analysis_version"]
assert BANK_PREDICTOR_SOURCE == VERSION_2_MAIN_TIMING_POLICY["bank_predictor_source"]
assert MACRO_PREDICTOR_SOURCE == VERSION_2_MAIN_TIMING_POLICY["macro_predictor_source"]
assert TARGET_WINDOW == VERSION_2_MAIN_TIMING_POLICY["target_window"]
assert REFERENCE_CALENDAR_START == VERSION_2_MAIN_TIMING_POLICY["reference_calendar_start"]
assert BANK_PREDICTOR_LAG_QUARTERS == VERSION_2_MAIN_TIMING_POLICY["bank_predictor_lag_quarters"]
assert ARCHIVED_V1_READ_ONLY == VERSION_2_MAIN_TIMING_POLICY["archived_v1_read_only"]

print("Authoritative V2 timing policy locked successfully.")
print(f"Policy keys verified: {len(VERSION_2_MAIN_TIMING_POLICY)}")
print(json.dumps(VERSION_2_MAIN_TIMING_POLICY, indent=2))


## 4. Write and validate the shared analysis `config.py`


In [ ]:
# Write and validate the shared main analysis `config.py`

REPRODUCIBILITY_POLICY = {
    "random_seed": 20260804,
}

DATA_SCOPE = {
    "expected_ffiec_start": "2001Q1",
    "expected_ffiec_end": "2026Q1",
    "eligible_bank_classes": ("N", "NM", "SM"),
}

CAPTURE_POLICY = {
    "primary": "distinct_failed_bank",
    "secondary": "positive_bank_quarter",
    "quarterly_top_risk_percentages": (1, 5, 10, 20),
}

HISTORY_POLICY = {
    "action": "retain_with_indicator",
    "required_consecutive_quarters": 5,
    "source_timing": "calculated_at_source_quarter_t_minus_1",
    "count_definition": (
        "Consecutive quarters observed using only information available up to the "
        "source quarter; reset when the immediately preceding calendar quarter is absent."
    ),
    "indicator_source_name": "insufficient_history_5q",
    "indicator_model_name": "insufficient_history_5q_lag1",
    "future_information_allowed": False,
}

MISSING_DATA_POLICY = {
    "strategy": "training_sample_median",
    "fit_scope": "training_rows_within_each_fold_only",
    "scope": "all_predictor_level_missing_values_when_exact_t_minus_1_report_exists",
    "missing_exact_predecessor_action": "drop_reference_observation",
    "apply_consistently_to": (
        "logistic_regression",
        "random_forest",
        "xgboost",
    ),
    "xgboost_native_missing_handling": False,
}

VALIDATION_FOLDS = (
    {
        "fold": 1,
        "train_start": "2001Q2",
        "train_end": "2008Q4",
        "purge_start": "2009Q1",
        "purge_end": "2009Q4",
        "validation_start": "2010Q1",
        "validation_end": "2012Q4",
    },
    {
        "fold": 2,
        "train_start": "2001Q2",
        "train_end": "2011Q4",
        "purge_start": "2012Q1",
        "purge_end": "2012Q4",
        "validation_start": "2013Q1",
        "validation_end": "2015Q4",
    },
    {
        "fold": 3,
        "train_start": "2001Q2",
        "train_end": "2014Q4",
        "purge_start": "2015Q1",
        "purge_end": "2015Q4",
        "validation_start": "2016Q1",
        "validation_end": "2018Q4",
    },
)

FINAL_TEST_DESIGN = {
    "train_start": "2001Q2",
    "train_end": "2018Q4",
    "purge_start": "2019Q1",
    "purge_end": "2019Q4",
    "test_start": "2020Q1",
    "test_end": "2025Q2",
}

CRISIS_ROBUSTNESS_DESIGN = {
    "train_start": "2001Q2",
    "train_end": "2006Q4",
    "purge_start": "2007Q1",
    "purge_end": "2007Q4",
    "evaluation_start": "2008Q1",
    "evaluation_end": "2012Q4",
}

target_horizon_match = re.search(r"plus_(\d+)_days", TARGET_WINDOW)
assert target_horizon_match is not None
TARGET_HORIZON_DAYS = int(target_horizon_match.group(1))
RANDOM_SEED = REPRODUCIBILITY_POLICY["random_seed"]

config_lines = [
    "from pathlib import Path",
    "import re",
    "",
    f"PROJECT_ROOT = Path({str(PROJECT_ROOT)!r})",
    "RAW_ROOT = PROJECT_ROOT / 'Raw Data'",
    "FFIEC_RAW_DIR = RAW_ROOT / 'FFIEC'",
    "FDIC_RAW_DIR = RAW_ROOT / 'FDIC'",
    "MACRO_RAW_DIR = RAW_ROOT / 'Macro'",
    "FDIC_FILE = FDIC_RAW_DIR / 'FDIC Data.xlsx'",
    "",
    "V2_ROOT = PROJECT_ROOT / 'Version 2 Main Analysis'",
    "V2_OUTPUT_ROOT = V2_ROOT / 'Notebook Outputs'",
    "V2_PROCESSED_DATA_DIR = V2_ROOT / 'Processed Data'",
    "V1_OUTPUT_ROOT = PROJECT_ROOT / 'Notebook Outputs'",
    "V1_PROCESSED_DATA_DIR = PROJECT_ROOT / 'Processed Data'",
    "",
    f"EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS = {pformat(EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS, sort_dicts=False)}",
    "",
    f"VERSION_2_MAIN_TIMING_POLICY = {pformat(VERSION_2_MAIN_TIMING_POLICY, sort_dicts=False)}",
    "",
    "assert tuple(VERSION_2_MAIN_TIMING_POLICY) == EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS",
    "assert set(VERSION_2_MAIN_TIMING_POLICY) == set(EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS)",
    "",
    "ANALYSIS_ROLE = VERSION_2_MAIN_TIMING_POLICY['analysis_role']",
    "ANALYSIS_VERSION = VERSION_2_MAIN_TIMING_POLICY['analysis_version']",
    "BANK_PREDICTOR_SOURCE = VERSION_2_MAIN_TIMING_POLICY['bank_predictor_source']",
    "MACRO_PREDICTOR_SOURCE = VERSION_2_MAIN_TIMING_POLICY['macro_predictor_source']",
    "TARGET_WINDOW = VERSION_2_MAIN_TIMING_POLICY['target_window']",
    "REFERENCE_CALENDAR_START = VERSION_2_MAIN_TIMING_POLICY['reference_calendar_start']",
    "BANK_PREDICTOR_LAG_QUARTERS = VERSION_2_MAIN_TIMING_POLICY['bank_predictor_lag_quarters']",
    "ARCHIVED_V1_READ_ONLY = VERSION_2_MAIN_TIMING_POLICY['archived_v1_read_only']",
    "",
    "assert ANALYSIS_ROLE == VERSION_2_MAIN_TIMING_POLICY['analysis_role']",
    "assert ANALYSIS_VERSION == VERSION_2_MAIN_TIMING_POLICY['analysis_version']",
    "assert BANK_PREDICTOR_SOURCE == VERSION_2_MAIN_TIMING_POLICY['bank_predictor_source']",
    "assert MACRO_PREDICTOR_SOURCE == VERSION_2_MAIN_TIMING_POLICY['macro_predictor_source']",
    "assert TARGET_WINDOW == VERSION_2_MAIN_TIMING_POLICY['target_window']",
    "assert REFERENCE_CALENDAR_START == VERSION_2_MAIN_TIMING_POLICY['reference_calendar_start']",
    "assert BANK_PREDICTOR_LAG_QUARTERS == VERSION_2_MAIN_TIMING_POLICY['bank_predictor_lag_quarters']",
    "assert ARCHIVED_V1_READ_ONLY == VERSION_2_MAIN_TIMING_POLICY['archived_v1_read_only']",
    "",
    f"REPRODUCIBILITY_POLICY = {pformat(REPRODUCIBILITY_POLICY, sort_dicts=False)}",
    "RANDOM_SEED = REPRODUCIBILITY_POLICY['random_seed']",
    "TARGET_HORIZON_DAYS = int(re.search(r'plus_(\\d+)_days', TARGET_WINDOW).group(1))",
    "",
    f"DATA_SCOPE = {pformat(DATA_SCOPE, sort_dicts=False)}",
    f"CAPTURE_POLICY = {pformat(CAPTURE_POLICY, sort_dicts=False)}",
    f"HISTORY_POLICY = {pformat(HISTORY_POLICY, sort_dicts=False)}",
    f"MISSING_DATA_POLICY = {pformat(MISSING_DATA_POLICY, sort_dicts=False)}",
    f"VALIDATION_FOLDS = {pformat(VALIDATION_FOLDS, sort_dicts=False)}",
    f"FINAL_TEST_DESIGN = {pformat(FINAL_TEST_DESIGN, sort_dicts=False)}",
    f"CRISIS_ROBUSTNESS_DESIGN = {pformat(CRISIS_ROBUSTNESS_DESIGN, sort_dicts=False)}",
]

CONFIG_PATH.write_text("\n".join(config_lines) + "\n", encoding="utf-8")

spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
v2_config = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = v2_config
spec.loader.exec_module(v2_config)

assert tuple(v2_config.VERSION_2_MAIN_TIMING_POLICY) == EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS
assert v2_config.VERSION_2_MAIN_TIMING_POLICY == VERSION_2_MAIN_TIMING_POLICY
assert v2_config.RANDOM_SEED == RANDOM_SEED
assert v2_config.TARGET_HORIZON_DAYS == TARGET_HORIZON_DAYS == 365
assert v2_config.VALIDATION_FOLDS[0]["train_start"] == "2001Q2"
assert v2_config.FINAL_TEST_DESIGN["test_end"] == "2025Q2"
assert v2_config.V2_OUTPUT_ROOT != v2_config.V1_OUTPUT_ROOT

print(f"Shared V2 configuration written and re-imported successfully:\n{CONFIG_PATH}")
print(f"Random seed retained for comparability: {RANDOM_SEED}")
print(f"Target horizon retained: {TARGET_HORIZON_DAYS} days after reference quarter t")


## 5. Record the Analysis change log


In [ ]:
# Record the earlier analysis → main analysis change log

version_change_log = pd.DataFrame(
    [
        {
            "change_id": "V2-01",
            "area": "analysis role",
            "version_1": "archived sensitivity analysis",
            "version_2": "main analysis",
            "reason": "Reporting availability and supervisor feedback; choice is unconditional on performance.",
        },
        {
            "change_id": "V2-02",
            "area": "bank predictor timing",
            "version_1": "contemporaneous reference-quarter bank predictors",
            "version_2": "exact immediately preceding calendar-quarter bank report, B(t-1)",
            "reason": "Reduce timing ambiguity between reported bank data and the failure window.",
        },
        {
            "change_id": "V2-03",
            "area": "missing predecessor",
            "version_1": "not applicable",
            "version_2": "drop reference row when exact t-1 report is absent; never use an older report",
            "reason": "Enforce an exact and auditable lag match.",
        },
        {
            "change_id": "V2-04",
            "area": "macro timing",
            "version_1": "one-quarter lag",
            "version_2": "retain existing one-quarter lag; do not shift again",
            "reason": "Keep both predictor groups measured at t-1 without double-lagging macro variables.",
        },
        {
            "change_id": "V2-05",
            "area": "reference calendar",
            "version_1": "began 2001Q1",
            "version_2": "begins 2001Q2; 2001Q1 supplies the first source report",
            "reason": "An exact t-1 bank source is required.",
        },
        {
            "change_id": "V2-06",
            "area": "development workflow",
            "version_1": "completed",
            "version_2": "repeat EDA, all validation folds, tuning and threshold selection",
            "reason": "V2 must be selected and fitted independently using the lagged panel.",
        },
        {
            "change_id": "V2-07",
            "area": "output storage",
            "version_1": str(V1_OUTPUT_ROOT),
            "version_2": str(V2_OUTPUT_ROOT),
            "reason": "Prevent any Version 1 file from being overwritten.",
        },
        {
            "change_id": "V2-08",
            "area": "comparison",
            "version_1": "full V1 result retained",
            "version_2": "require common-sample V1 comparison as sensitivity",
            "reason": "Separate sample-composition changes from model-timing changes as far as possible.",
        },
        {
            "change_id": "V2-09",
            "area": "effective predictor-date horizon",
            "version_1": "shorter predictor-date-to-failure horizon",
            "version_2": "derive realised horizon from source_report_date to failure_date",
            "reason": "The outcome window stays anchored at t while bank predictors move to t-1.",
        },
    ]
)

version_change_log_path = NB01_OUTPUT_DIR / "version_change_log.csv"
version_change_log.to_csv(version_change_log_path, index=False)
display(version_change_log)
print(f"Version-change log saved to:\n{version_change_log_path}")


## 6. Define file-audit helpers


In [ ]:
# Define file-audit helpers

def classify_source(path):
    if is_relative_to(path, FFIEC_RAW_DIR):
        return "FFIEC"
    if is_relative_to(path, FDIC_RAW_DIR):
        return "FDIC"
    if is_relative_to(path, MACRO_RAW_DIR):
        return "Macro"
    return "Other"


def extract_years(path_text):
    years = re.findall(r"\b(20\d{2})\b", path_text)
    return sorted({int(year) for year in years if 2001 <= int(year) <= 2026})


def quarter_end_date(year, quarter):
    month = quarter * 3
    day = calendar.monthrange(year, month)[1]
    return datetime(year, month, day).date()


def extract_report_dates(path_text):
    dates = set()

    for year, month, day in re.findall(
        r"(?<!\d)(20\d{2})[-_](\d{2})[-_](\d{2})(?!\d)", path_text
    ):
        try:
            dates.add(datetime(int(year), int(month), int(day)).date())
        except ValueError:
            pass

    for value in re.findall(r"(?<!\d)(\d{8})(?!\d)", path_text):
        for date_format in ("%m%d%Y", "%Y%m%d"):
            try:
                parsed = datetime.strptime(value, date_format).date()
                if 2001 <= parsed.year <= 2026:
                    dates.add(parsed)
                    break
            except ValueError:
                continue

    for year, quarter in re.findall(
        r"(?<!\d)(20\d{2})[\s_-]*Q([1-4])(?!\d)",
        path_text,
        flags=re.IGNORECASE,
    ):
        dates.add(quarter_end_date(int(year), int(quarter)))

    return sorted(dates)


def extract_schedule_hint(filename):
    stem = Path(filename).stem
    for pattern in (
        r"\bSchedule[\s_-]+([A-Z0-9]+)\b",
        r"\bBulk[\s_-]+([A-Z0-9]+)\b",
    ):
        match = re.search(pattern, stem, flags=re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None


def detect_text_structure(path):
    sample_text = None
    encoding_used = None
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as file:
                sample_text = file.readline()
            encoding_used = encoding
            break
        except UnicodeDecodeError:
            continue

    if sample_text is None:
        raise UnicodeError("Unable to decode text file with supported encodings.")

    delimiter_counts = {
        "\t": sample_text.count("\t"),
        ",": sample_text.count(","),
        "|": sample_text.count("|"),
        ";": sample_text.count(";"),
    }
    delimiter = max(delimiter_counts, key=delimiter_counts.get)
    if delimiter_counts[delimiter] == 0:
        column_count = 1
        delimiter_name = "undetected"
    else:
        column_count = len(next(csv.reader([sample_text], delimiter=delimiter)))
        delimiter_name = {"\t": "tab", ",": "comma", "|": "pipe", ";": "semicolon"}[delimiter]

    line_count = 0
    last_byte = None
    with path.open("rb") as file:
        while True:
            block = file.read(1024 * 1024)
            if not block:
                break
            line_count += block.count(b"\n")
            last_byte = block[-1:]
    if path.stat().st_size > 0 and last_byte != b"\n":
        line_count += 1

    return {
        "row_count": max(line_count - 1, 0),
        "column_count": column_count,
        "sheet_count": np.nan,
        "sheet_dimensions": None,
        "encoding": encoding_used,
        "delimiter": delimiter_name,
        "archive_member_count": np.nan,
        "embedded_name_hints": None,
        "dimension_method": "header_columns_and_physical_lines",
    }


def inspect_excel_structure(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    sheet_dimensions = {}
    total_data_rows = 0
    maximum_columns = 0
    for worksheet in workbook.worksheets:
        rows = worksheet.max_row or 0
        columns = worksheet.max_column or 0
        sheet_dimensions[worksheet.title] = {
            "rows_including_header": rows,
            "columns": columns,
        }
        total_data_rows += max(rows - 1, 0)
        maximum_columns = max(maximum_columns, columns)
    sheet_names = list(workbook.sheetnames)
    workbook.close()
    return {
        "row_count": total_data_rows,
        "column_count": maximum_columns,
        "sheet_count": len(sheet_names),
        "sheet_dimensions": json.dumps(sheet_dimensions, sort_keys=True),
        "encoding": None,
        "delimiter": None,
        "archive_member_count": np.nan,
        "embedded_name_hints": None,
        "dimension_method": "openpyxl_used_dimensions",
    }


def inspect_zip_file(path):
    with zipfile.ZipFile(path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise ValueError(f"Corrupted ZIP member: {bad_member}")
        members = [name for name in archive.namelist() if not name.endswith("/")]
    return {
        "row_count": np.nan,
        "column_count": np.nan,
        "sheet_count": np.nan,
        "sheet_dimensions": None,
        "encoding": None,
        "delimiter": None,
        "archive_member_count": len(members),
        "embedded_name_hints": ";".join(members),
        "dimension_method": "archive_member_count_only",
    }


def inspect_file(path):
    result = {
        "readable": True,
        "read_error": None,
        "row_count": np.nan,
        "column_count": np.nan,
        "sheet_count": np.nan,
        "sheet_dimensions": None,
        "encoding": None,
        "delimiter": None,
        "archive_member_count": np.nan,
        "embedded_name_hints": None,
        "dimension_method": "binary_open_test",
    }
    try:
        suffix = path.suffix.lower()
        if suffix in {".csv", ".txt", ".tsv", ".dat"}:
            result.update(detect_text_structure(path))
        elif suffix in {".xlsx", ".xlsm"}:
            result.update(inspect_excel_structure(path))
        elif suffix == ".zip":
            result.update(inspect_zip_file(path))
        else:
            with path.open("rb") as file:
                file.read(1)
    except Exception as error:
        result["readable"] = False
        result["read_error"] = f"{type(error).__name__}: {error}"
    return result


def calculate_sha256(path):
    sha256 = hashlib.sha256()
    with path.open("rb") as file:
        while True:
            block = file.read(1024 * 1024)
            if not block:
                break
            sha256.update(block)
    return sha256.hexdigest()


## 7. Scan every raw file and create the manifest


In [ ]:
# Scan every raw file and create the manifest

required_paths = (FFIEC_RAW_DIR, FDIC_RAW_DIR, MACRO_RAW_DIR, FDIC_FILE)
missing_required_paths = [str(path) for path in required_paths if not path.exists()]

if missing_required_paths:
    raise FileNotFoundError(
        "Required raw-data paths are missing:\n" + "\n".join(missing_required_paths)
    )

raw_files = sorted(
    path
    for path in RAW_ROOT.rglob("*")
    if path.is_file()
    and not any(part.startswith(".") for part in path.relative_to(RAW_ROOT).parts)
)
if not raw_files:
    raise ValueError("No raw files were found under the Raw Data folder.")

print(f"Files found: {len(raw_files):,}")
print("Inspecting files. Large FFIEC files may take several minutes.")

manifest_records = []
for path in tqdm(raw_files):
    relative_path = path.relative_to(RAW_ROOT)
    stat = path.stat()
    inspection = inspect_file(path)
    searchable_names = str(relative_path)
    if inspection.get("embedded_name_hints"):
        searchable_names += ";" + inspection["embedded_name_hints"]

    years = extract_years(searchable_names)
    report_dates = extract_report_dates(searchable_names)
    manifest_records.append(
        {
            "source": classify_source(path),
            "relative_path": str(relative_path),
            "parent_folder": path.parent.name,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "schedule_hint": extract_schedule_hint(path.name),
            "year_hints": ";".join(map(str, years)),
            "report_dates_found": ";".join(map(str, report_dates)),
            "size_bytes": stat.st_size,
            "size_mb": round(stat.st_size / (1024 ** 2), 4),
            "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
            **inspection,
        }
    )

manifest = pd.DataFrame(manifest_records)
manifest.insert(0, "file_id", range(1, len(manifest) + 1))
manifest["sha256"] = pd.NA

print("Raw-file structural inspection completed.")
display(manifest.head(20))


## 8. Detect exact duplicate files


In [ ]:
# Detect exact duplicate files

same_size_groups = manifest.groupby("size_bytes").groups
candidate_group_count = sum(
    1 for size, indices in same_size_groups.items() if size > 0 and len(indices) > 1
)
print(f"Same-size groups requiring hashes: {candidate_group_count:,}")

for size_bytes, row_indices in tqdm(same_size_groups.items(), total=len(same_size_groups)):
    row_indices = list(row_indices)
    if size_bytes > 0 and len(row_indices) > 1:
        for row_index in row_indices:
            full_path = RAW_ROOT / manifest.loc[row_index, "relative_path"]
            try:
                manifest.loc[row_index, "sha256"] = calculate_sha256(full_path)
            except Exception as error:
                manifest.loc[row_index, "readable"] = False
                manifest.loc[row_index, "read_error"] = (
                    f"HashReadError: {type(error).__name__}: {error}"
                )

duplicate_mask = manifest["sha256"].notna() & manifest.duplicated("sha256", keep=False)
duplicate_files = (
    manifest.loc[
        duplicate_mask,
        ["source", "relative_path", "filename", "size_bytes", "sha256"],
    ]
    .sort_values(["sha256", "relative_path"])
    .reset_index(drop=True)
)

if not duplicate_files.empty:
    group_map = {
        file_hash: group_number
        for group_number, file_hash in enumerate(duplicate_files["sha256"].unique(), start=1)
    }
    duplicate_files.insert(0, "duplicate_group", duplicate_files["sha256"].map(group_map))
else:
    duplicate_files.insert(0, "duplicate_group", pd.Series(dtype="Int64"))

exact_duplicate_groups = int(duplicate_files["sha256"].nunique())
print(f"Exact duplicate groups: {exact_duplicate_groups:,}")
display(duplicate_files.head(20))


## 9. Check FFIEC year and reporting-quarter coverage


In [ ]:
# Check FFIEC year and reporting-quarter coverage

ffiec_manifest = manifest.loc[manifest["source"].eq("FFIEC")].copy()
expected_years = list(range(2001, 2027))
expected_quarters = pd.period_range(
    DATA_SCOPE["expected_ffiec_start"],
    DATA_SCOPE["expected_ffiec_end"],
    freq="Q",
)

detected_years = set()
for year_text in ffiec_manifest["year_hints"].dropna():
    for year in str(year_text).split(";"):
        if year:
            detected_years.add(int(year))

ffiec_year_coverage = pd.DataFrame(
    [
        {
            "year": year,
            "year_detected": year in detected_years,
            "file_count": int(
                ffiec_manifest["year_hints"].str.contains(
                    rf"(?:^|;){year}(?:;|$)", regex=True, na=False
                ).sum()
            ),
        }
        for year in expected_years
    ]
)

year_file_totals = []
for year in expected_years:
    mask = ffiec_manifest["year_hints"].str.contains(
        rf"(?:^|;){year}(?:;|$)", regex=True, na=False
    )
    year_file_totals.append(
        {
            "year": year,
            "total_size_mb": round(ffiec_manifest.loc[mask, "size_mb"].sum(), 4),
            "readable_files": int(ffiec_manifest.loc[mask, "readable"].sum()),
            "unreadable_files": int((~ffiec_manifest.loc[mask, "readable"]).sum()),
        }
    )
ffiec_year_coverage = ffiec_year_coverage.merge(
    pd.DataFrame(year_file_totals), on="year", how="left", validate="one_to_one"
)

date_records = []
for _, row in ffiec_manifest.iterrows():
    for date_text in str(row["report_dates_found"]).split(";"):
        if date_text and date_text != "nan":
            date_records.append(
                {
                    "report_date": date_text,
                    "relative_path": row["relative_path"],
                    "filename": row["filename"],
                }
            )

ffiec_date_inventory = pd.DataFrame(
    date_records,
    columns=["report_date", "relative_path", "filename"],
)
if not ffiec_date_inventory.empty:
    ffiec_date_inventory["report_date"] = pd.to_datetime(ffiec_date_inventory["report_date"])
    ffiec_date_inventory = ffiec_date_inventory.sort_values(
        ["report_date", "relative_path"]
    ).reset_index(drop=True)

expected_report_dates = pd.DatetimeIndex(
    [period.end_time.normalize() for period in expected_quarters]
)
detected_report_dates = (
    set(ffiec_date_inventory["report_date"].dt.normalize())
    if not ffiec_date_inventory.empty
    else set()
)

ffiec_quarter_coverage = pd.DataFrame(
    {
        "quarter": expected_quarters.astype(str),
        "expected_report_date": expected_report_dates,
    }
)
ffiec_quarter_coverage["date_detected_in_path_or_archive_name"] = (
    ffiec_quarter_coverage["expected_report_date"].isin(detected_report_dates)
)

missing_ffiec_years = ffiec_year_coverage.loc[
    ~ffiec_year_coverage["year_detected"], "year"
].astype(int).tolist()
missing_ffiec_quarters = ffiec_quarter_coverage.loc[
    ~ffiec_quarter_coverage["date_detected_in_path_or_archive_name"], "quarter"
].tolist()

print(f"Expected FFIEC years: {len(expected_years)}")
print(f"Years detected: {len(detected_years)}")
print(f"Missing years: {missing_ffiec_years}")
print(f"Expected quarter-end dates: {len(expected_quarters)}")
print(f"Quarter-end dates detected in path/archive names: {len(detected_report_dates)}")
display(ffiec_year_coverage)


## 10. Build the audit summary and save all Notebook 01 outputs


In [ ]:
# Build the audit summary and save all Notebook 01 outputs

missing_macro_files = [
    filename for filename in EXPECTED_MACRO_FILES
    if not (MACRO_RAW_DIR / filename).exists()
]
unreadable_files = manifest.loc[~manifest["readable"]].copy()
empty_files = manifest.loc[manifest["size_bytes"].eq(0)].copy()

audit_rows = [
    {
        "check": "V2 output isolation from V1",
        "status": "PASS",
        "detail": f"All writes are restricted to {V2_ROOT}",
    },
    {
        "check": "Authoritative V2 timing-policy key set",
        "status": "PASS",
        "detail": f"Exactly {len(EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS)} ordered keys verified",
    },
    {
        "check": "Shared V2 config import",
        "status": "PASS",
        "detail": str(CONFIG_PATH),
    },
    {
        "check": "FFIEC raw folder",
        "status": "PASS" if FFIEC_RAW_DIR.exists() else "FAIL",
        "detail": str(FFIEC_RAW_DIR),
    },
    {
        "check": "FDIC failure file",
        "status": "PASS" if FDIC_FILE.exists() else "FAIL",
        "detail": str(FDIC_FILE),
    },
    {
        "check": "Macro raw folder",
        "status": "PASS" if MACRO_RAW_DIR.exists() else "FAIL",
        "detail": str(MACRO_RAW_DIR),
    },
    {
        "check": "FFIEC year coverage: 2001–2026",
        "status": "PASS" if not missing_ffiec_years else "FAIL",
        "detail": (
            "All expected years detected"
            if not missing_ffiec_years
            else f"Missing years: {missing_ffiec_years}"
        ),
    },
    {
        "check": "FFIEC quarter-end dates in path/archive names",
        "status": "PASS" if detected_report_dates and not missing_ffiec_quarters else "WARNING",
        "detail": (
            "All expected quarter-end dates detected"
            if detected_report_dates and not missing_ffiec_quarters
            else (
                f"{len(missing_ffiec_quarters)} expected dates were not detected in names; "
                "Notebook 02 must verify quarter coverage from file contents."
            )
        ),
    },
    {
        "check": "Required macro files",
        "status": "PASS" if not missing_macro_files else "FAIL",
        "detail": (
            "All four required macro files found"
            if not missing_macro_files
            else f"Missing files: {missing_macro_files}"
        ),
    },
    {
        "check": "Unreadable files",
        "status": "PASS" if unreadable_files.empty else "FAIL",
        "detail": f"{len(unreadable_files)} unreadable file(s)",
    },
    {
        "check": "Empty files",
        "status": "PASS" if empty_files.empty else "WARNING",
        "detail": f"{len(empty_files)} empty file(s)",
    },
    {
        "check": "Exact duplicate files",
        "status": "PASS" if exact_duplicate_groups == 0 else "WARNING",
        "detail": f"{exact_duplicate_groups} exact duplicate group(s)",
    },
]
audit_summary = pd.DataFrame(audit_rows)
overall_status = "PASS" if not audit_summary["status"].eq("FAIL").any() else "REVIEW REQUIRED"

output_paths = {
    "raw_file_manifest": NB01_OUTPUT_DIR / "raw_file_manifest_v2.csv",
    "ffiec_year_coverage": NB01_OUTPUT_DIR / "ffiec_year_coverage_v2.csv",
    "ffiec_quarter_coverage": NB01_OUTPUT_DIR / "ffiec_quarter_coverage_v2.csv",
    "ffiec_reporting_date_inventory": NB01_OUTPUT_DIR / "ffiec_reporting_date_inventory_v2.csv",
    "exact_duplicate_files": NB01_OUTPUT_DIR / "exact_duplicate_files_v2.csv",
    "unreadable_files": NB01_OUTPUT_DIR / "unreadable_files_v2.csv",
    "empty_files": NB01_OUTPUT_DIR / "empty_files_v2.csv",
    "audit_summary": NB01_OUTPUT_DIR / "audit_summary_v2.csv",
    "audit_report_json": NB01_OUTPUT_DIR / "audit_report_v2.json",
    "audit_report_markdown": NB01_OUTPUT_DIR / "audit_report_v2.md",
}

for path in output_paths.values():
    assert is_relative_to(path, V2_ROOT), f"Unsafe output path outside V2 root: {path}"

manifest.to_csv(output_paths["raw_file_manifest"], index=False)
ffiec_year_coverage.to_csv(output_paths["ffiec_year_coverage"], index=False)
ffiec_quarter_coverage.to_csv(output_paths["ffiec_quarter_coverage"], index=False)
ffiec_date_inventory.to_csv(output_paths["ffiec_reporting_date_inventory"], index=False)
duplicate_files.to_csv(output_paths["exact_duplicate_files"], index=False)
unreadable_files.to_csv(output_paths["unreadable_files"], index=False)
empty_files.to_csv(output_paths["empty_files"], index=False)
audit_summary.to_csv(output_paths["audit_summary"], index=False)

audit_report = {
    "notebook": "01_Raw_Data_Audit_and_File_Manifest_V2",
    "analysis_version": ANALYSIS_VERSION,
    "analysis_role": ANALYSIS_ROLE,
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "overall_status": overall_status,
    "policy_keys_exact": True,
    "policy_key_count": len(VERSION_2_MAIN_TIMING_POLICY),
    "random_seed": RANDOM_SEED,
    "target_horizon_days": TARGET_HORIZON_DAYS,
    "reference_calendar_start": REFERENCE_CALENDAR_START,
    "total_raw_files": int(len(manifest)),
    "ffiec_files": int(manifest["source"].eq("FFIEC").sum()),
    "fdic_files": int(manifest["source"].eq("FDIC").sum()),
    "macro_files": int(manifest["source"].eq("Macro").sum()),
    "missing_ffiec_years": missing_ffiec_years,
    "missing_ffiec_quarters_from_names": missing_ffiec_quarters,
    "missing_macro_files": missing_macro_files,
    "unreadable_file_count": int(len(unreadable_files)),
    "empty_file_count": int(len(empty_files)),
    "exact_duplicate_groups": exact_duplicate_groups,
    "raw_root_read_only": str(RAW_ROOT),
    "v1_output_root_read_only": str(V1_OUTPUT_ROOT),
    "config_path": str(CONFIG_PATH),
    "output_folder": str(NB01_OUTPUT_DIR),
}
output_paths["audit_report_json"].write_text(
    json.dumps(audit_report, indent=2) + "\n", encoding="utf-8"
)

audit_table_lines = [
    "| Check | Status | Detail |",
    "|---|---|---|",
]
for row in audit_summary.itertuples(index=False):
    safe_check = str(row.check).replace("|", "\\|")
    safe_status = str(row.status).replace("|", "\\|")
    safe_detail = str(row.detail).replace("|", "\\|").replace("\n", " ")
    audit_table_lines.append(f"| {safe_check} | {safe_status} | {safe_detail} |")

markdown_lines = [
    "# Notebook 01 V2 Audit Report",
    "",
    f"- Run time (UTC): {audit_report['run_time_utc']}",
    f"- Overall status: **{overall_status}**",
    f"- Analysis version: `{ANALYSIS_VERSION}`",
    f"- Raw files scanned: {len(manifest):,}",
    f"- FFIEC / FDIC / macro files: "
    f"{audit_report['ffiec_files']:,} / {audit_report['fdic_files']:,} / {audit_report['macro_files']:,}",
    f"- Missing FFIEC years: {missing_ffiec_years}",
    f"- Missing required macro files: {missing_macro_files}",
    f"- Unreadable files: {len(unreadable_files):,}",
    f"- Empty files: {len(empty_files):,}",
    f"- Exact duplicate groups: {exact_duplicate_groups:,}",
    f"- V2 policy keys: {len(VERSION_2_MAIN_TIMING_POLICY)} (exact key assertion passed)",
    f"- V2 output root: `{V2_ROOT}`",
    "",
    "## Checks",
    "",
    *audit_table_lines,
    "",
]
output_paths["audit_report_markdown"].write_text(
    "\n".join(markdown_lines), encoding="utf-8"
)

display(audit_summary)
print(f"\nOVERALL NOTEBOOK 01 V2 STATUS: {overall_status}")
print(f"Outputs saved under:\n{NB01_OUTPUT_DIR}")


## 11. Final execution gate


In [ ]:
# Final execution gate

print("=" * 78)
print("NOTEBOOK 01 V2 — FINAL AUDIT RESULT")
print("=" * 78)
print(f"Overall status: {overall_status}")
print(f"Total raw files: {len(manifest):,}")
print(f"FFIEC files: {manifest['source'].eq('FFIEC').sum():,}")
print(f"FDIC files: {manifest['source'].eq('FDIC').sum():,}")
print(f"Macro files: {manifest['source'].eq('Macro').sum():,}")
print(f"Unreadable files: {len(unreadable_files):,}")
print(f"Empty files: {len(empty_files):,}")
print(f"Exact duplicate groups: {exact_duplicate_groups:,}")
print(f"Missing FFIEC years: {missing_ffiec_years}")
print(f"Missing macro files: {missing_macro_files}")
print(f"Policy key assertion: PASS ({len(VERSION_2_MAIN_TIMING_POLICY)} exact keys)")
print(f"Shared config: {CONFIG_PATH}")
print(f"Output folder: {NB01_OUTPUT_DIR}")

assert CONFIG_PATH.exists(), "V2 config.py was not created."
assert version_change_log_path.exists(), "Version-change log was not created."
assert all(path.exists() for path in output_paths.values()), "One or more V2 audit outputs are missing."
assert not audit_summary["status"].eq("FAIL").any(), (
    "Notebook 01 V2 audit contains FAIL results. Review audit_summary_v2.csv "
    "before running Notebook 02."
)

if audit_summary["status"].eq("WARNING").any():
    print("\nNotebook 01 completed with warnings. Review warning rows before Notebook 02.")
else:
    print("\nNotebook 01 completed successfully with no warnings.")
print("The project may now proceed to Notebook 02.")
